# Titanic Survival Prediction - Feature Engineering

## Part 1: Data Cleaning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load data
train = pd.read_csv('../data/train.csv')
print("Original shape:", train.shape)
print("\nMissing values:\n", train.isnull().sum())

### 1.1 Missing Value Handling

In [ ]:
# Drop Cabin (77% missing)
train.drop('Cabin', axis=1, inplace=True)

# Impute Age with median
train['Age'].fillna(train['Age'].median(), inplace=True)

# Impute Embarked with mode
train['Embarked'].fillna(train['Embarked'].mode()[0], inplace=True)

# Create indicator for missing Age (though none after imputation)
train['Age_Missing'] = 0

print("Missing values after handling:\n", train.isnull().sum())

### 1.2 Outlier Handling

In [ ]:
# Function to detect outliers using IQR
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return len(outliers)

print(f"Age outliers: {detect_outliers_iqr(train, 'Age')}")
print(f"Fare outliers: {detect_outliers_iqr(train, 'Fare')}")

# Cap Fare outliers at 95th percentile
fare_95 = train['Fare'].quantile(0.95)
train['Fare'] = np.where(train['Fare'] > fare_95, fare_95, train['Fare'])

# Cap Age outliers at 95th percentile (though age outliers are valid)
age_95 = train['Age'].quantile(0.95)
train['Age'] = np.where(train['Age'] > age_95, age_95, train['Age'])

print("Outliers handled")

### 1.3 Data Consistency

In [ ]:
# Fix Sex values
train['Sex'] = train['Sex'].map({'male': 'Male', 'female': 'Female'})

# Remove duplicates
train.drop_duplicates(inplace=True)

print(f"Shape after cleaning: {train.shape}")

# Save cleaned dataset
train.to_csv('../data/train_cleaned.csv', index=False)
print("Saved to ../data/train_cleaned.csv")

## Part 2: Feature Engineering

### 2.1 Create Derived Features

In [ ]:
# FamilySize
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1

# IsAlone
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

# Title extraction from Name
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Mme': 'Mrs',
    'Capt': 'Rare', 'Sir': 'Rare'
}
train['Title'] = train['Title'].map(title_mapping)

# Age groups
def age_group(age):
    if age < 12:
        return 'Child'
    elif age < 18:
        return 'Teen'
    elif age < 60:
        return 'Adult'
    else:
        return 'Senior'
train['AgeGroup'] = train['Age'].apply(age_group)

# Fare per person
train['FarePerPerson'] = train['Fare'] / train['FamilySize']

print("Derived features created:")
print(f"  - FamilySize: {train['FamilySize'].unique()[:5]}...")
print(f"  - IsAlone: {train['IsAlone'].unique()}")
print(f"  - Title: {train['Title'].unique()}")
print(f"  - AgeGroup: {train['AgeGroup'].unique()}")
print(f"  - FarePerPerson: range {train['FarePerPerson'].min():.2f} to {train['FarePerPerson'].max():.2f}")

### 2.2 Categorical Encoding

In [ ]:
# One-hot encoding for nominal features
train = pd.get_dummies(train, columns=['Sex', 'Embarked', 'Title', 'AgeGroup'], drop_first=True)

# Drop original text columns
train.drop(['Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)

print(f"Shape after encoding: {train.shape}")
print(f"\nColumns now: {list(train.columns)[:10]}...")

### 2.3 Interaction Features

In [ ]:
# Interaction features
train['Pclass_Fare'] = train['Pclass'] * train['Fare']

# Age × Title_Mr interaction (if Title_Mr exists)
if 'Title_Mr' in train.columns:
    train['Age_Title_Mr'] = train['Age'] * train['Title_Mr']

print("Interaction features created")

### 2.4 Feature Transformations

In [ ]:
# Log transform skewed features
train['Fare_log'] = np.log1p(train['Fare'])
train['Age_log'] = np.log1p(train['Age'])

# Standardize features
features_to_scale = ['Age', 'Fare', 'FarePerPerson', 'Fare_log', 'Age_log', 'FamilySize']
scaler = StandardScaler()
train[features_to_scale] = scaler.fit_transform(train[features_to_scale])

print(f"Final shape after feature engineering: {train.shape}")
print("Features transformed and scaled")

## Part 3: Feature Selection

### 3.1 Correlation Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))
corr_matrix = train.corr()
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

# Remove highly correlated features (correlation > 0.85)
corr_threshold = 0.85
corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > corr_threshold:
            corr_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

print("Highly correlated pairs:", corr_pairs)

# Drop Fare_log (highly correlated with Fare)
if 'Fare_log' in train.columns:
    train.drop('Fare_log', axis=1, inplace=True)
    print("Dropped Fare_log due to high correlation with Fare")

### 3.2 Feature Importance using Random Forest

In [ ]:
X = train.drop('Survived', axis=1)
y = train['Survived']

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance (Top 15):")
print(feature_importance.head(15))

### 3.3 Final Feature Selection

In [ ]:
# Keep top 10 features based on importance
selected_features = feature_importance.head(10)['feature'].tolist()
selected_features_with_target = selected_features + ['Survived']

print("\nSelected features for final model:")
for i, f in enumerate(selected_features, 1):
    print(f"  {i}. {f}")

# Final dataset with selected features
final_df = train[selected_features_with_target]
final_df.to_csv('../data/train_selected_features.csv', index=False)
print("\nSaved to ../data/train_selected_features.csv")

## Visualizations & Key Findings

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Survival by Sex
if 'Sex_Male' in train.columns:
    sns.barplot(data=train, x='Sex_Male', y='Survived', ax=axes[0,0])
    axes[0,0].set_title('Survival Rate by Sex (1=Male)')

# Survival by Pclass
sns.barplot(data=train, x='Pclass', y='Survived', ax=axes[0,1])
axes[0,1].set_title('Survival Rate by Pclass')

# Survival by IsAlone
sns.barplot(data=train, x='IsAlone', y='Survived', ax=axes[0,2])
axes[0,2].set_title('Survival Rate: IsAlone vs Not Alone')

# Age distribution by survival
sns.kdeplot(data=train, x='Age', hue='Survived', fill=True, ax=axes[1,0])
axes[1,0].set_title('Age Distribution by Survival')

# Fare distribution by survival
sns.kdeplot(data=train, x='Fare', hue='Survived', fill=True, ax=axes[1,1])
axes[1,1].set_title('Fare Distribution by Survival')

# Feature importance bar plot
sns.barplot(data=feature_importance.head(10), x='importance', y='feature', ax=axes[1,2])
axes[1,2].set_title('Top 10 Feature Importance')

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\n" + "="*50)
print("KEY FINDINGS")
print("="*50)
print(f"1. Overall survival rate: {y.mean()*100:.1f}%")
print("2. Women had significantly higher survival rate than men")
print("3. First class passengers had highest survival rate (approx 63%)")
print("4. Passengers traveling alone had lower survival rate")
print("5. Fare was strongly correlated with survival (wealthier passengers survived more)")
print("6. Age showed U-shaped pattern: children and elderly had moderate survival, young adults lowest")

# Survival by Title if exists
title_cols = [c for c in train.columns if c.startswith('Title_')]
if title_cols:
    print("\n7. Survival by Title:")
    for col in title_cols:
        survival = train[train[col]==1]['Survived'].mean() if (train[col]==1).any() else 0
        print(f"   - {col.replace('Title_', '')}: {survival*100:.1f}%")